# Model Setup and Loading

As mentioned in the first notebook, `01_lens_mechanics_and_intuition`, we will be using the GPT-2 model, which can be easily loaded using the `load_model_and_tokenizer` function.

In [ ]:
import sys

# Add project root in the path
sys.path.append('..')

In [ ]:
from src.model_utils import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer('gpt2')

# Data Extraction

For the comparative analysis section, we use the data provided in `prompts_benchmark.txt`, which consists of prompts divided into four categories: `Knowledge`, `Arithmetic`, `Sequences`, and `Syntax`. There are a total of 50 data points for each category, which is sufficient for the purposes of this project.

In [ ]:
import pandas as pd

# Load data
df_prompts = pd.read_csv('prompts_benchmark.txt', sep='\t')

df_prompts = df_prompts.sample(frac=1).reset_index(drop=True) # Disorganize data

df_prompts.head()

In the data there is a column called `target`, this represents the expected prediction, but it doesn't necessarily mean that this is the model’s actual prediction. Therefore, the next step is to filter out the data points where the target matches the model’s prediction.

In [ ]:
import torch

correct_predictions = 0
total_prompts = len(df_prompts)

model.eval() # Evaluation mode

correct_prompts = []

for index, row in df_prompts.iterrows():
    prompt = row['prompt']
    target = row['target']
    
    # Take tokenizer input
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Calculate output
    with torch.no_grad():
        outputs = model(**inputs)

    next_token_id = outputs.logits[0, -1, :].argmax(dim=-1).item()
    predicted_text = tokenizer.decode(next_token_id) # Get next word
    
    # Check if target matches with the prediction
    if predicted_text == target:
        correct_predictions += 1
        correct_prompts.append(row)
    
# Make new data frame
df_filtred = pd.DataFrame(correct_prompts).reset_index(drop=True)

print(f"Correct predictions: {correct_predictions} / {total_prompts}")

# Analysis of the Kullback-Leibler Divergence

The KL divergence measures the dissimilarity between two probability distributions. This metric is calculated as
$$
D_{\text{KL}}(P || Q) = \sum_i P \ln{\frac{P(i)}{Q(i)}}.
$$
In this case, $P$ is the distribution of the final layer and $Q$ is the distribution of the current layer we are comparing.

It is clear that the KL divergence of the final layer will always match and is 0. Furthermore, this is exactly the loss function used to train the Tuned Lens, so it is expected that the KL divergence of the Tuned Lens will always be smaller than that of the Logit Lens.

In [ ]:
from src.notebooks_utils import calculate_kl_divergence

from src.logit_lens import get_all_logit_lens_logits
from src.tuned_lens import get_all_tuned_lens_logits, SingleLayerTunedLens

import torch.nn as nn

num_layers = model.config.n_layer

# Total sum of kl divergence per layer
kl_div_sums_logit = {layer: 0.0 for layer in range(1, num_layers + 1)} 
kl_div_sums_tuned = {layer: 0.0 for layer in range(1, num_layers + 1)}

total_prompts = len(df_filtred)

# Import lenses

# Structure from lenses
hidden_size = model.config.n_embd
device = 'cpu'

lenses = nn.ModuleDict({
    str(layer_idx): SingleLayerTunedLens(hidden_size)
    for layer_idx in range(1, num_layers)
})

# Load weights saved
lenses.load_state_dict(torch.load('..\checkpoints/all_tuned_lenses.pt', map_location=device))
lenses.to(device)

lenses.eval()

# 
model.eval()
vocab_size = tokenizer.vocab_size

for index, row in df_filtred.iterrows():
    prompt = row['prompt']
    
    # Use function to get logits and final logits
    layer_logits_dict_logit = get_all_logit_lens_logits(model, tokenizer, prompt)
    layer_logits_dict_tuned = get_all_tuned_lens_logits(model, tokenizer, lenses, prompt)
    
    final_logits = layer_logits_dict_logit[num_layers]
    
    # Calculate kl div for each layer
    for layer_idx in range(1, num_layers + 1):
        layer_logits_logit = layer_logits_dict_logit[layer_idx]
        layer_logits_tuned = layer_logits_dict_tuned[layer_idx]
        
        # KL divergence for Logit Lens
        kl_val_logit = calculate_kl_divergence(layer_logits_logit.unsqueeze(0), final_logits.unsqueeze(0)) # Adjust the dimensions
        
        # KL divergence for Tuned Lens
        kl_val_tuned = calculate_kl_divergence(layer_logits_tuned.unsqueeze(0), final_logits.unsqueeze(0)) # Adjust the dimensions
        
        # Storage kl values
        kl_div_sums_logit[layer_idx] += kl_val_logit.item()
        kl_div_sums_tuned[layer_idx] += kl_val_tuned.item()
        
# Get average
kl_div_avg_logit = {layer: (kl_div_sums_logit[layer] / total_prompts) for layer in range(1, num_layers + 1)}
kl_div_avg_tuned = {layer: (kl_div_sums_tuned[layer] / total_prompts) for layer in range(1, num_layers + 1)}

In [ ]:
import matplotlib.pyplot as plt

layers = list(kl_div_avg_logit.keys())
logit_values = list(kl_div_avg_logit.values())
tuned_values = list(kl_div_avg_tuned.values())

# Graph
plt.figure(figsize=(10, 6))

plt.plot(layers, logit_values, label='Logit Lens', marker='o', color="#007900")
plt.plot(layers, tuned_values, label='Tuned Lens', marker='s', color="#570079")

plt.xlabel('Layers')
plt.ylabel('KL Divergence')

plt.grid()

plt.legend()
plt.show()

# Analysis of the Mean Reciprocal Rank

Another way to measure how effective the techniques are is by analyzing the position of the correct word in a given layer. For example, in the final layer, the correct word usually occupies the first position, whereas in earlier layers (such as layer 1) it may appear in lower positions, such as the seventh.
Using this idea, we employ the Mean Reciprocal Rank (MRR), defined as
$$
\text{MRR} = \frac{1}{|Q|}\sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i},
$$
where $Q$ is the set of queries, and $\text{rank}_i$ corresponds to the ranking position of the correct word for query $i$.

In this case, MRR allows for a better comparison of techniques such as Logit Lens and Tuned Lens, as it is particularly useful for analyzing how the quality of internal representations evolves across the model’s layers.

In [ ]:
from src.notebooks_utils import calculate_reciprocal_rank

rr_logit = {layer:0.0 for layer in range(1, num_layers + 1)}
rr_tuned = {layer:0.0 for layer in range(1, num_layers + 1)}

for index, row in df_filtred.iterrows():
    prompt = row['prompt'] # Get prompt
    target = row['target'] # Get target
    
    target_token_id = tokenizer.encode(target)[0] # Get target id
    
    # Use function to get logits
    layer_logits_dict_logit = get_all_logit_lens_logits(model, tokenizer, prompt)
    layer_logits_dict_tuned = get_all_tuned_lens_logits(model, tokenizer, lenses, prompt)
    
    for layer in range(1, num_layers + 1):
        actual_layer_logit = layer_logits_dict_logit[layer]
        actual_layer_tuned = layer_logits_dict_tuned[layer]
        
        reciprocal_rank_logit, _ = calculate_reciprocal_rank(actual_layer_logit, target_token_id)
        reciprocal_rank_tuned, _ = calculate_reciprocal_rank(actual_layer_tuned, target_token_id)
        
        rr_logit[layer] += reciprocal_rank_logit
        rr_tuned[layer] += reciprocal_rank_tuned
        
mrr_logit = {layer: (rr_logit[layer] / total_prompts) for layer in range(1, num_layers + 1)}
mrr_tuned = {layer: (rr_tuned[layer] / total_prompts) for layer in range(1, num_layers + 1)}

In [ ]:
layers = list(mrr_logit.keys())
mrr_logit_values = list(mrr_logit.values())
mrr_tuned_values = list(mrr_tuned.values())

# Graph
plt.figure(figsize=(10, 6))

plt.plot(layers, mrr_logit_values, label='Logit Lens', marker='o',color="#007900")
plt.plot(layers, mrr_tuned_values, label='Tuned Lens', marker='s',color="#570079")

plt.xlabel('Layers')
plt.ylabel('Mean Reciprocal Rank (MRR)')

plt.grid()

plt.legend()
plt.show()

# Analysis of the Initial Token Offering Layer

The goal of this section is to visualize the cumulative distribution function, which we calculate simply by summing, for each layer, the number of times the model correctly predicts the target word in the first position.

This is particularly relevant when comparing techniques such as Logit Lens and Tuned Lens, as it allows us to identify not only which one achieves better final results, but also which one is able to concentrate the probability on the correct answer sooner.

In [ ]:
from src.logit_lens import run_logit_lens
from src.tuned_lens import run_tuned_lens

CDF_logit = {layer:0.0 for layer in range(1, num_layers + 1)}
CDF_tuned = {layer:0.0 for layer in range(1, num_layers + 1)}

num_prompts = df_filtred.shape[0]

for index, row in df_filtred.iterrows():
    prompt = row['prompt']
    target = row['target']
    
    # Use function to get logits
    layer_dict_logit = run_logit_lens(model, tokenizer, prompt, top_k=1)
    layer_dict_tuned = run_tuned_lens(model, tokenizer, lenses, prompt, top_k=1)
    
    for layer in range(1, num_layers + 1):
        [(word_predicted_logit, _)] = layer_dict_logit[layer]
        [(word_predicted_tuned, _)] = layer_dict_tuned[layer]
        
        # Acummulate the correct predictions
        if target == word_predicted_logit: CDF_logit[layer] += 1.0 / num_prompts
            
        if target == word_predicted_tuned: CDF_tuned[layer] += 1.0 / num_prompts

In [ ]:
import numpy as np

layers = list(CDF_logit.keys())
CDF_logit_values = list(CDF_logit.values())
CDF_tuned_values = list(CDF_tuned.values())

x = np.arange(len(layers))
width = 0.35

# Graph
fig, ax = plt.subplots(figsize=(10, 6))

bars_logit = ax.bar(x - width/2, CDF_logit_values, width, label='Logit Lens', color='#007900')
bars_tuned = ax.bar(x + width/2, CDF_tuned_values, width, label='Tuned Lens', color='#570079')

ax.set_xlabel('Layers')
ax.set_ylabel('Cumulative Hit Rate')

ax.set_axisbelow(True)
ax.set_xticks(x)

ax.yaxis.grid(linestyle='--')

ax.legend()
plt.show()

# Analysis of the Different Categories of Prompts

To conclude our analysis, we compare the different categories of prompts included in the benchmark. Specifically, we aim to determine, on average, at which layer the target word first appears in the top position of the ranking.

This metric allows us to analyze how “quickly” each type of prompt is resolved by the model—that is, at what point in the internal processing does the representation become informative enough to correctly identify the answer. In this way, we can observe whether there are categories that require deeper reasoning compared to those whose answer can be inferred at earlier stages of the model.

In [ ]:
first_layer_logit_list = []
first_layer_tuned_list = []

for index, row in df_filtred.iterrows():
    prompt = row['prompt']
    target = row['target']
    
    # Get logits of the prompt
    logits_logit_dict = get_all_logit_lens_logits(model, tokenizer, prompt)
    logits_tuned_dict = get_all_tuned_lens_logits(model, tokenizer, lenses, prompt)
    
    hit_logit = num_layers # By deafault this is the last layer
    hit_tuned = num_layers
    
    # This is for Logit Lens
    for layer in range(1, num_layers):
        pred_token = tokenizer.decode(logits_logit_dict[layer].argmax(dim=-1).item())
        
        # Search the minimal layer than pred is equal to the target
        if pred_token == target:
            hit_logit = layer
            break
            
    # This is for Tuned Lens
    for layer in range(1, num_layers):
        pred_token = tokenizer.decode(logits_tuned_dict[layer].argmax(dim=-1).item())
        
        # Search the minimal layer than pred is equal to the target
        if pred_token == target:
            hit_tuned = layer
            break 
            
    # Save this in the list
    first_layer_logit_list.append(hit_logit)
    first_layer_tuned_list.append(hit_tuned)

In [ ]:
# Add the last list to the DataFrame
df_filtred['acquisition_layer_logit'] = first_layer_logit_list
df_filtred['acquisition_layer_tuned'] = first_layer_tuned_list

# Get average of the minimal layer for each category
df_averages = df_filtred.groupby('category')[['acquisition_layer_logit', 'acquisition_layer_tuned']].mean()

# Things for the graphs
categories = df_averages.index.tolist()
values_logit = df_averages['acquisition_layer_logit'].tolist()
values_tuned = df_averages['acquisition_layer_tuned'].tolist()

# Close the radar
values_logit += values_logit[:1]
values_tuned += values_tuned[:1]

# Calculate the angle for the radar
N = len(categories)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1] 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.spines['polar'].set_visible(False)
ax.yaxis.grid(False) 

for level in np.arange(0, num_layers + 1, step=2):
    ax.plot(angles, [level]*len(angles), color='lightgray', linestyle='solid', linewidth=1.0, zorder=0)

# Logit Lens
ax.plot(angles, values_logit, color='#007900', linewidth=1.5, linestyle='solid', label='Logit Lens', zorder=2)
ax.fill(angles, values_logit, color='#007900', alpha=0.25, zorder=2)

# Tuned Lens
ax.plot(angles, values_tuned, color='#570079', linewidth=1.5, linestyle='solid', label='Tuned Lens', zorder=2)
ax.fill(angles, values_tuned, color='#570079', alpha=0.25, zorder=2)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([cat.title() for cat in categories])
ax.tick_params(axis='x', pad=20)

ax.set_ylim(0, num_layers) 
ax.set_yticks(np.arange(0, num_layers + 1, step=1)) 
ax.set_yticklabels([])

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()